In [41]:
import pandas as pd
import polars as pl

In [42]:
filepath = "../src/commandes_ecommerce.csv"
df_pd = pd.read_csv(filepath)
df_pd.head()

,commande_id,date_commande,client_id,age_client,ville,canal,categorie,prix_unitaire,quantite,remise_pct,mode_paiement,delai_livraison_jours,note_client,retour
0,CMD-00001,2025-01-01,C285,24,Bordeaux,Magasin,Mode,43.36,1,10,Carte bancaire,NaN,NaN,False
1,CMD-00002,2025-01-01,C194,57,Lille,Site web,Sport,44.41,2,10,Carte bancaire,4.0,5.0,False
2,CMD-00003,2025-01-01,C166,26,Toulouse,Application mobile,Mode,27.33,1,10,Carte bancaire,4.0,4.0,False
3,CMD-00004,2025-01-02,C213,38,Paris,Site web,Électronique,113.62,1,30,Carte bancaire,7.0,4.0,False
4,CMD-00005,2025-01-02,C215,34,Strasbourg,Site web,Maison,61.00,2,20,Carte cadeau,2.0,NaN,False


In [43]:
df_pl = pl.read_csv(filepath)
df_pl.head()

commande_id,date_commande,client_id,age_client,ville,canal,categorie,prix_unitaire,quantite,remise_pct,mode_paiement,delai_livraison_jours,note_client,retour
str,str,str,i64,str,str,str,f64,i64,i64,str,f64,f64,bool
"""CMD-00001""","""2025-01-01""","""C285""",24,"""Bordeaux""","""Magasin""","""Mode""",43.36,1,10,"""Carte bancaire""",null,null,false
"""CMD-00002""","""2025-01-01""","""C194""",57,"""Lille""","""Site web""","""Sport""",44.41,2,10,"""Carte bancaire""",4.0,5.0,false
"""CMD-00003""","""2025-01-01""","""C166""",26,"""Toulouse""","""Application mobile""","""Mode""",27.33,1,10,"""Carte bancaire""",4.0,4.0,false
"""CMD-00004""","""2025-01-02""","""C213""",38,"""Paris""","""Site web""","""Électronique""",113.62,1,30,"""Carte bancaire""",7.0,4.0,false
"""CMD-00005""","""2025-01-02""","""C215""",34,"""Strasbourg""","""Site web""","""Maison""",61.0,2,20,"""Carte cadeau""",2.0,null,false


In [44]:
%timeit a = df_pd[(df_pd["age_client"] > 30) & (df_pd["ville"] == "Paris")]
%timeit b = df_pl.filter((pl.col("age_client") > 30) & (pl.col("ville") == "Paris"))

1.11 ms ± 100 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
2.98 ms ± 69.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [45]:
grand_pd = pd.concat([df_pd] * 1000, ignore_index=True)
grand_pl = pl.concat([df_pl] * 1000).with_row_index("index")

In [46]:
%timeit a = grand_pd[(grand_pd["age_client"] > 30) & (grand_pd["ville"] == "Paris")]
%timeit a = grand_pl.filter((pl.col("age_client") > 30) & (pl.col("ville") == "Paris"))

86.8 ms ± 1.51 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
16.8 ms ± 362 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [28]:
%timeit grand_pd.groupby("ville").agg(age_moy=("age_client", "mean"), age_max=("age_client", "max"), n=("age_client", "size"))
%timeit grand_pl.group_by("ville").agg(pl.col("age_client").mean().alias("age_moy"), pl.col("age_client").max().alias("age_max"), pl.len().alias("n"))

106 ms ± 3.87 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
20.4 ms ± 756 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [29]:
%timeit grand_pd.assign(moy_ville=grand_pd.groupby("ville")["age_client"].transform("mean"))
%timeit grand_pl.with_columns(pl.col("age_client").mean().over("ville").alias("moy_ville"))

97.8 ms ± 5.63 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
27.2 ms ± 2.78 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [31]:
import numpy as np

In [32]:
def tranche(age):
    return "jeune" if age < 30 else ("adulte" if age < 60 else "senior")

# pandas: row-by-row apply vs vectorized
%timeit grand_pd["age_client"].apply(tranche)
%timeit np.select([grand_pd["age_client"] < 30, grand_pd["age_client"] < 60], ["jeune", "adulte"], "senior")

# Polars: map_elements (Python function, slow) vs native expression
%timeit grand_pl.select(pl.col("age_client").map_elements(tranche, return_dtype=pl.String))
%timeit grand_pl.select(pl.when(pl.col("age_client") < 30).then(pl.lit("jeune")).when(pl.col("age_client") < 60).then(pl.lit("adulte")).otherwise(pl.lit("senior")))

115 ms ± 1.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
14.3 ms ± 301 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
188 ms ± 11.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
17.1 ms ± 810 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%timeit grand_pd["age_client"].apply(lambda a: "majeur" if a >= 18 else "mineur")
%timeit np.where(grand_pd["age_client"] >= 18, "majeur", "mineur")

%timeit grand_pl.select(pl.col("age_client").map_elements(lambda a: "majeur" if a >= 18 else "mineur", return_dtype=pl.String))
%timeit grand_pl.select(pl.when(pl.col("age_client") >= 18).then(pl.lit("majeur")).otherwise(pl.lit("mineur")))

88.6 ms ± 3.08 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
8.21 ms ± 49.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
152 ms ± 8.26 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
884 μs ± 15 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [34]:
a = grand_pd["age_client"].apply(lambda a: "majeur" if a >= 18 else "mineur")
b = np.where(grand_pd["age_client"] >= 18, "majeur", "mineur")
c = grand_pl.select(pl.col("age_client").map_elements(lambda a: "majeur" if a >= 18 else "mineur", return_dtype=pl.String))
d = grand_pl.select(pl.when(pl.col("age_client") >= 18).then(pl.lit("majeur")).otherwise(pl.lit("mineur")))

In [39]:
%timeit df_pd["age_client"].apply(lambda a: "majeur" if a >= 18 else "mineur")
%timeit np.where(df_pd["age_client"] >= 18, "majeur", "mineur")
%timeit df_pl.select(pl.col("age_client").map_elements(lambda a: "majeur" if a >= 18 else "mineur", return_dtype=pl.String))
%timeit df_pl.select(pl.when(pl.col("age_client") >= 18).then(pl.lit("majeur")).otherwise(pl.lit("mineur")))

282 μs ± 18.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
110 μs ± 7.94 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.55 ms ± 62.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.02 ms ± 15.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
